# Lineare Regression und Feature Engineering

Irgendwo in einem Labor steht eine Maschine, die Zahlen produziert. Wir kennen ihre Eingaben ($x_1$ und $x_2$) und ihre verrauschte Ausgabe ($y$) — aber die wahre innere Funktion ist uns unbekannt. Können wir sie rekonstruieren, ohne uns vom Rauschen täuschen zu lassen?

Spoiler: Zu viel Komplexität ist gefährlich.

## Benötigte Bibliotheken

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## Teil 1: Datensatz

### Daten laden und visualisieren

Der Code im nächsten Teil ladet und visualisiert den Datensatz. Sieht er linear aus? Welche Art von Funktion könnte dahinterstecken?

In [ ]:
file_dataset = 'Datensätze/mystery.csv'
dataset = np.loadtxt(file_dataset, delimiter=',', skiprows=1)

# Visualisierung, ein Diagramm pro Merkmal
_, axes = plt.subplots(1, dataset.shape[1] - 1, figsize=(10, 4.5), dpi=100, sharey='all')
for index_feature in range(dataset.shape[1] - 1):
    ax = axes[index_feature]
    ax.scatter(dataset[:, index_feature], dataset[:, -1], alpha=0.6, edgecolors='k', linewidths=0.5)
    ax.set(xlabel=f'$x_{index_feature + 1}$')
    if index_feature == 0:
        ax.set_ylabel('y (Ausgabe)')
    ax.grid(True, alpha=0.3)
plt.suptitle(f'Die mysteriöse Maschine; {dataset.shape[0]} Punkte')
plt.tight_layout()
plt.show()
del axes, index_feature, ax

### Trainings- und Validierungssatz

Wir werden alle Beobachtungen zufällig in einen Trainings- und einen Validierungsdatensatz aufteilen.

In [ ]:
def train_test_split(x: np.ndarray, y: np.ndarray, train_ratio: float) -> \
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Teilt einen Datensatz in einen Trainings- und einen Testsatz auf.

    :param x: Eingabewerte als Numpy-Array der Form `(n, p)`, wobei
              `n` die Anzahl an Beobachtungen ist und `p` - die Anzahl an Merkmalen.
    :param y: Ausgabewerte für die Beobachtungen im Trainingssatz.
    :param train_ratio: Anteil der Beobachtungen zur Bildung des Trainingssatzes.
    :return: Tupel `(x_train, y_train, x_test, y_test)`.
    """
    # Elemente mischen
    # generator = np.random.default_rng(19751979)
    # indices = generator.permutation(x.shape[0])

    # Höchste Werte für x[:, 0] dem Testsatz zurodnen
    indices = x[:, 0].argsort()

    # Trainings- und Testsatz erzeugen
    train_size = int(train_ratio * indices.size)
    indices_train = indices[:train_size]
    indices_test = indices[train_size:]
    x_train, y_train = x[indices_train, :], y[indices_train]
    x_test, y_test = x[indices_test, :], y[indices_test]
    return x_train, y_train, x_test, y_test


# Trainings- und Validierungssatz erstellen
x_train, y_train, x_val, y_val =\
    train_test_split(dataset[:, :-1], dataset[:, -1], train_ratio=0.9)

print(f'Form von x_train und y_train: {x_train.shape}, {y_train.shape}')
print(f'    Form von x_val und y_val: {x_val.shape}, {y_val.shape}')

## Teil 2: Polynomiale Features erzeugen

Wir definieren mit Hilfe unserer 2 Merkmale $x_1, x_2$ neue Features vom Grad $d$:

$x_{1}^d, x_1^{d-1} x_2, ..., x_2^d$.

Damit kann ein lineares Modell nichtlineare Zusammenhänge lernen.


#### Übung: Polynomiale Feature-Matrix erstellen

Schreiben Sie eine Funktion, die aus einer Matrix $X$ (mit den Merkmalenwerten von $x_1$ und $x_2$) eine neue Matrix erzeugt, die alle Merkmalkombinationen vom Grad $D$ beinhaltet.

Hier sind alle Terme eines Polynoms, die Grad 2 haben: $(x_1^2, \; x_1 x_2, \; x_2^2)$.

Hier sind alle Terme eines Polynoms, die Grad 3 haben: $(x_1^3, \; x_1^2 x_2, \; x_1 x_2^2, \; x_2^3)$.

Hinweis: Die Bias-Spalte (Einsen) fügen wir später separat hinzu.

In [ ]:
def create_polynomial_features(x: np.ndarray, degree: int) -> np.ndarray:
    """
    Erzeugt eine Matrix mit polynomialen Features vom Grad $D$.

    :param x: Eingabewerte für 2 Merkmale als Numpy-Array der Form `(n, 2)`.
    :param degree: Polynomgrad als ganze Zahl >= 1.
    :return: Array der Form `(n, degree + 1)` mit den Werten der neuen Features.
    """
    if degree == 1:
        return x.copy()
    result = np.empty((x.shape[0], degree + 1), dtype=x.dtype)
    for i in range(degree + 1):
        result[:, i] = (x[:, 0] ** i) * (x[:, 1] ** (degree - i))
    return result

Testen wir die neuen Merkmalenwerte für Grad $3$.

Das erwartete Ergebniss ist:

`[[ -0.0777998   0.51011809 -3.34474477 21.9308388 ]`<br>
` [31.41689522 25.39318298 20.52442602 16.58917922]]`

In [ ]:
features3 = create_polynomial_features(x_train, degree=3)
print(features3[:2, :])

### Feature-Normalisierung

Zur Erinnerung: $x^8$ kann *riesig* werden, wenn $x > 3$. Dadurch explodieren die Gradienten und die Normalengleichung wird numerisch instabil. Z-Score-Normalisierung löst das Problem:

$z = (x - \mu) / \sigma$

Wir berechnen $\mu$ und $\sigma$ nur auf den Trainingsdaten und wenden die gleiche Transformation auf die Testdaten an.

In [ ]:
def normalize_features(x_train: np.ndarray, x_val: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Z-Score-Normalisierung.
    
    :param x_train: Trainingssatz mit normierten Merkmale als Numpy-Array der Form `(n, p)`, wobei
                    `n` die Anzahl an Beobachtungen ist und `p` - die Anzahl an Merkmalen. Spalte
                    `0` muss die Bias-Spalte sein.
    :param x_val: Validierungssatz mit normierten Merkmale als Numpy-Array der Form `(m, p)`, wobei
                  `m` die Anzahl an Beobachtungen ist und `p` - die Anzahl an Merkmalen. Spalte `0`
                  muss die Bias-Spalte sein.
    :return: Tupel `(x_train_norm, x_val_norm, mu, sigma)`.
    """
    mu = x_train.mean(axis=0)
    sigma = x_train.std(axis=0)
    sigma[sigma == 0] = 1  # Division durch Null vermeiden
    x_train_norm = (x_train - mu) / sigma
    x_val_norm = (x_val - mu) / sigma
    return x_train_norm, x_val_norm, mu, sigma

## Teil 3: Analytische Lösung der linearen Regression

Für die lineare Regression mit mehreren Features (inklusive Bias) lautet die Normalengleichung:

$\mathbf{w} = (X^T X)^{-1} X^T y$

Wir verwenden `np.linalg.solve`, was numerisch stabiler ist als die explizite Inverse zu berechnen:

$X^T X \cdot \mathbf{w} = X^T y \Rightarrow \mathbf{w} = \text{solve}(X^T X, X^T y)$

In [ ]:
def fit_linear_regression(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """
    Berechnet die optimalen Gewichte mit der Normalengleichung.

    :param x: Trainingssatz mit normierten Merkmalen als Numpy-Array der Form `(n, p)`, wobei
              `n` die Anzahl an Beobachtungen ist und `p` - die Anzahl an Merkmalen. Spalte
              `0` muss die Bias-Spalte sein.
    :param y: Ausgabewerte für die Beobachtungen im Trainingssatz.
    :return: Gewichtsvektor als Numpy-Array der Form `(p,)`.
    """
    # Kleine Regularisierung für numerische Stabilität
    regularisation = 1e-9 * np.eye(x.shape[1])
    return np.linalg.solve(x.T @ x + regularisation, x.T @ y)


def mse(y_true, y_predicted):
    """Mean Squared Error."""
    return np.mean((y_true - y_predicted) ** 2)

## Teil 4: Polynome verschiedener Grade trainieren

Wir fitten Polynome vom Grad 1 (linear) bis Grad 6 inklusiv und schauen, welches Modell am besten funktioniert.

In einer Schleife fügen wir iterativ die neuen Merkmale zum Modell hinzu, die dem höheren Grad des Polynoms entsprechen. Das Modell wird dann trainiert (mit dem Trainingssatz) und evaluiert (mit dem Validierungssatz).

In [ ]:
degree_max = 7

errors_pol = np.empty((degree_max, 2), dtype=np.float64)  # Trainings- und Validierungsloss für alle Polynoms
feature_count = np.empty(degree_max, dtype=np.uint32)  # Anzahl an Merkmalen pro Modell

# Am Anfag beinhalten unsere Trainings- und Validierungssatz nur die "Bias"-Spalte.
x_train_current = np.ones((x_train.shape[0], 1), dtype=x_train.dtype)
x_val_current = np.ones((x_val.shape[0], 1), dtype=x_val.dtype)

for degree in range(1, degree_max + 1):

    # Features erzeugen
    x_train_addition = create_polynomial_features(x_train, degree)
    x_val_addition = create_polynomial_features(x_val, degree)

    # Normalisieren
    x_train_addition, x_val_addition, _, _ =\
        normalize_features(x_train_addition, x_val_addition)

    # Neue Features hinzufügen
    x_train_current = np.hstack((x_train_current, x_train_addition))
    x_val_current = np.hstack((x_val_current, x_val_addition))

    # Modell berechnen
    feature_count[degree - 1] = x_train_current.shape[1]
    w = fit_linear_regression(x_train_current, y_train)

    # Verlustwerte berechnen
    y_hat_train = x_train_current @ w
    y_hat_val = x_val_current @ w
    errors_pol[degree - 1, 0] = mse(y_train, y_hat_train)
    errors_pol[degree - 1, 1] = mse(y_val, y_hat_val)
    del x_train_addition, x_val_addition, w, y_hat_train, y_hat_val

# Die Variablen:
# x_train_current, y_train, x_val_current, y_val
# speichern einen 'kompleten' Datensatz, der die Merkmale
# x_1 und x_2 beinhaltet, sowie alle Terme eines Polynoms von Grad 6.

print(f'Form der x_train_current: {x_train_current.shape}')
print(f'Form der x_val_current: {x_val_current.shape}')

## Teil 5: Verlustwerte visualisieren

Im Array `errors_pol` sind die Verlustwerte für alle ausprobierte Modelle gespeichert. Wir können sie in ein Diagram visualisieren, das die Trainings- und Validierungsverluste als zwei Kurven zeigt. Die einfachen Modelle werden auf der linken Seite angezeigt und die komplexen auf der rechten Seite.

So ein Diagram visualisiert das [Verzerrung-Varianz-Dilemma](https://de.wikipedia.org/wiki/Verzerrung-Varianz-Dilemma), in diesem Fall mit der Modellfamilie der *polynomiellen Regression* bis Grad `degree_max`.

In [ ]:
def plot_train_validation(errors: np.ndarray, xlabel: str, model_labels: list[str]):
    """
    Visualisiert die Kurven der Trainings- und Validierungsverluste.
    
    :param errors: Verlustwerte als Numpy-Array der Form `(n, 2)`. Reihen in dieser Matrix bezeichnen
                   Komplexitätsstufen; die 2 Spalten - Trainings- und Validierungsverluste. 
    :param xlabel: Beschreibung der Modellkomplexität, dienst als die Beschriftung der x-Achse.
    :param model_labels: Namen von allen Komplexitätsstufen als `list` von Länge `n`.
    """
    plot_width = max(1 + errors.shape[0] * 0.2, 5)
    _, ax = plt.subplots(figsize=(plot_width, 5), dpi=100)
    xs = np.arange(errors.shape[0])
    ax.plot(xs, errors[:, 0], 'o--', c='#1F77B4', label='Training')
    ax.plot(xs, errors[:, 1], 'o-', c='#FF7F0E', label='Validierung')
    ax.set(axisbelow=True, xlim=[xs[0] - 0.2, xs[-1] + 0.2], xticks=xs, xticklabels=model_labels)
    ax.set(xlabel=xlabel, ylabel='Verlustwert', yscale='log')
    ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
    ax.legend()
    plt.tight_layout()
    plt.show()


model_labels = [str(i) for i in range(1, errors_pol.shape[0] + 1)]
plot_train_validation(errors_pol, 'Polynomgrad', model_labels)

## Vorwärtsgerichtete schrittweise Auswahl

In der Realität haben wir oft viele Merkmale und wissen nicht, welche nützlich sind. *Forward Stepwise Selection* (Vorwärtsgerichtete schrittweise Auswahl) ist die folgende Strategie:

1. Starte ohne Features (nur Bias).
2. Für jedes noch nicht gewählte Feature: Probiere es hinzuzufügen und berechne den Trainingsfehler.
3. Wähle das Feature, das den Trainingsfehler am meisten verbessert.
4. Wiederhole, bis alle Features getestet sind.

So können wir beobachten, in welcher Reihenfolge die Merkmale nützlich sind und ab wann das Hinzufügen zu Überanpassung führt.

### Übung: Forward Stepwise Selection — die innere Schleife

Vervollständigen Sie den Code in der Funktion `forward_stepwise_selection`. Die folgenden Schritte müssen in der inneren Schleife implementiert werden:

1. Temporäre Trainings- und Validierungssatz mit allen ausgewählten + das neue Merkmal bauen.
2. Lineares Modell trainieren.
3. Trainings- und Validierungsloss berechnen.
4. Den Trainingsloss mit den alternativen Modellen vergleichen.

In [ ]:
def forward_stepwise_selection(x_train: np.ndarray, y_train: np.ndarray, x_val: np.ndarray, y_val: np.ndarray):
    """
    Vorwärtsgerichtete schrittweise Auswahl.
    
    :param x_train: Trainingssatz mit normierten Merkmale als Numpy-Array der Form `(n, p)`, wobei
                    `n` die Anzahl an Beobachtungen ist und `p` - die Anzahl an Merkmalen. Spalte
                    `0` muss die Bias-Spalte sein.
    :param y_train: Ausgabewerte für die Beobachtungen im Trainingssatz.
    :param x_val: Validierungssatz mit normierten Merkmale als Numpy-Array der Form `(m, p)`, wobei
                  `m` die Anzahl an Beobachtungen ist und `p` - die Anzahl an Merkmalen. Spalte `0`
                  muss die Bias-Spalte sein.
    :param y_val: Ausgabewerte für die Beobachtungen im Validierungssatz.
    :return: Tupel `(selected, errors)`. Hier ist:
             `selected` eine `list` der Länge `p`, die die Indizes der schrittweise ausgewählten
             Features beinhaltet. Das erste Element ist `0`;
             `errors` eine Numpy-Array der Form `(p - 1, 2)`, die die Trainings- und
             Validierungsverluste aller Modelle beinhaltet.
    """
    selected = [0]  # Indizes der gewählten Features
    remaining = list(range(1, x_train.shape[1]))  # Indizes der noch verfügbaren Features
    errors = np.empty((len(remaining), 2), dtype=np.float64)

    while len(remaining) != 0:
        index_best, loss_best = 0, 0.0
        for i, index_new_feature in enumerate(remaining):

            # Trainings- und Validierungssatz mit dem neuen Merkmal bauen
            current_features = np.array(selected + [index_new_feature], dtype=np.uint32)
            x_train_current = x_train[:, current_features]
            x_val_current = x_val[:, current_features]

            # Lineares Modell trainieren
            w = fit_linear_regression(x_train_current, y_train)

            # Trainings- und Validierungsloss berechnen
            y_hat_train = x_train_current @ w
            y_hat_val = x_val_current @ w
            loss_train = mse(y_train, y_hat_train)
            loss_val = mse(y_val, y_hat_val)

            # Validierungsloss mit den alternativen Modellen vergleichen
            if i == 0 or loss_train < loss_best:
                index_best, loss_best = index_new_feature, loss_train
                errors[len(selected) - 1, 0] = loss_train
                errors[len(selected) - 1, 1] = loss_val

        # Das Feature, das zur größten Verbesserung führt, hinzufügen
        selected.append(index_best)
        remaining.remove(index_best)
        del index_best, loss_best, i, index_new_feature

    return selected, errors

Wir können diesen Ansatz mit den kompletten Datensatz anwenden.

In [ ]:
selected, errors_ffs = forward_stepwise_selection(x_train_current, y_train, x_val_current, y_val)

clabels = [str(i + 1) for i in range(errors_ffs.shape[0])]
plot_train_validation(errors_ffs, 'Merkmale', clabels)

### Hausaufgabe

1. Erweitern Sie die Funktionalität der Funktion `create_polynomial_features` derart, dass sie auch Markmalbeschreibungen (Feature-Labels) zurückgibt.

2. Verwenden Sie diese Labels, um festzustellen, welche die 7 besten Features im Rahmen der Forward Stepwise Selection sind.

3. Wie sieht das Vorhersagemodell mit diesen ausgewählen Features aus?